# 🧬 Qwen-JEPA 0.5B: Arquitectura de Predicción en Espacio Latente

## Technical Implementation: Joint-Embedding Predictive Architecture

**Versión:** 1.0.0-RC1  
**Arquitectura Base:** Qwen 2.5 (0.5B)  
**Paradigma:** JEPA + VICReg + SIGReg  
**Hardware Target:** Google Colab Free Tier (Tesla T4 16GB)

---

### 🎯 Objetivo Arquitectónico

Transformar un LLM generativo en un **modelo de mundo** que aprende a predecir representaciones abstractas en espacio latente, eliminando el desperdicio computacional de la predicción token-a-token.

### 🔬 Innovaciones Clave

1. **Dual-Tower Asimétrica:** Context Encoder (Student) + Target Encoder (Teacher)
2. **VICReg Loss:** Prevención de colapso dimensional mediante regularización de Varianza-Invariancia-Covarianza
3. **SIGReg Weighting:** Ponderación dinámica basada en contenido informativo
4. **EMA Scheduling:** Actualización coseno del Teacher para convergencia estable
5. **Block Masking:** Predicción de bloques contiguos (no tokens aleatorios)


## 📦 Paso 1: Configuración del Entorno (Zero-Cost Stack)


In [ ]:
# ============================================================================
# 🔧 INSTALACIÓN DE DEPENDENCIAS CON RESOLUCIÓN DE CONFLICTOS
# ============================================================================
# Estrategia: Instalación ordenada para prevenir incompatibilidades
# - Fase 1: Resolver conflicto fsspec (base del ecosistema GCS)
# - Fase 2: Transformers (evitar versión 4.46.0 yanked)
# - Fase 3: Stack ML core
# - Fase 4: Herramientas de monitoreo

print("🔧 Iniciando instalación ordenada de dependencias...\n")

# FASE 1: Resolver conflicto fsspec (crítico para gcsfs en Colab)
print("[1/4] Actualizando fsspec a versión compatible...")
!pip install -q --upgrade 'fsspec>=2025.3.0'

# FASE 2: Transformers (evitar 4.46.0 yanked, usar versión estable)
print("[2/4] Instalando transformers (versión estable)...")
!pip install -q 'transformers>=4.45.0,<4.46.0' --upgrade

# FASE 3: Core ML stack
print("[3/4] Instalando datasets, accelerate...")
!pip install -q 'datasets>=3.0.0,<3.2.0' 'accelerate>=1.0.0'

# FASE 4: Monitoring & optimization tools
print("[4/4] Instalando wandb, bitsandbytes...")
!pip install -q 'wandb>=0.18.0' 'bitsandbytes>=0.44.0'

# PyTorch (ya viene preinstalado en Colab, pero verificamos versión)
import torch
print(f"\n✅ Dependencias instaladas")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA disponible: {torch.cuda.is_available()}")

# Verificación de compatibilidad fsspec-gcsfs
try:
    import fsspec
    import gcsfs
    print(f"   fsspec: {fsspec.__version__}")
    print(f"   gcsfs: {gcsfs.__version__}")
    print("   ✅ Compatibilidad gcsfs-fsspec verificada")
except ImportError as e:
    print(f"   ⚠️  Advertencia en verificación: {e}")

In [ ]:
# Imports del núcleo computacional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torch.cuda.amp import GradScaler, autocast

import wandb
import numpy as np
import math
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, Tuple, Optional

from transformers import AutoModel, AutoTokenizer, AutoConfig
from datasets import load_dataset

# Configuración de precisión y reproducibilidad
torch.set_float32_matmul_precision('high')
torch.manual_seed(42)
np.random.seed(42)

# Detección de hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Hardware detectado: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Autenticación WandB para monitoreo
wandb.login()

# Inicialización del proyecto
wandb.init(
    project="qwen-jepa-0.5b",
    name="jepa-vicreg-sigreg-v1",
    config={
        "architecture": "JEPA",
        "base_model": "Qwen/Qwen2.5-0.5B",
        "regularization": "VICReg + SIGReg",
        "hidden_dim": 1024,
        "predictor_hidden": 4096,
        "predictor_depth": 3,
    }
)

## 🏗️ Paso 2: Arquitectura del Latent Predictor

### Especificación Técnica

```
Input:  h_x ∈ ℝ^1024 (Context embedding)
Output: z_pred ∈ ℝ^1024 (Predicted target embedding)

Architecture:
  Linear(1024 → 4096) → BatchNorm → GELU
  ↓ [3x Residual Blocks]
  Linear(4096 → 4096) → BatchNorm → GELU → Residual
  ↓
  Linear(4096 → 1024)
```

**⚠️ Detalle Crítico:** BatchNorm (no LayerNorm) es esencial para VICReg.


In [ ]:
class LatentPredictor(nn.Module):
    """The Bridge: Maps context embeddings to target space.
    
    Critical Design Decisions:
    - BatchNorm1d (NOT LayerNorm) for VICReg compatibility
    - Residual connections for gradient flow
    - GELU activation for smooth gradients
    """
    
    def __init__(self, input_dim: int = 1024, hidden_dim: int = 4096, num_blocks: int = 3):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.input_norm = nn.BatchNorm1d(hidden_dim)
        
        # Residual blocks
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
            )
            for _ in range(num_blocks)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, input_dim)
        
        # Inicialización Xavier
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, h_context: torch.Tensor) -> torch.Tensor:
        """Forward pass con conexiones residuales.
        
        Args:
            h_context: [batch_size, seq_len, input_dim] o [batch_size, input_dim]
        
        Returns:
            z_pred: [batch_size, input_dim] predicción en espacio latente
        """
        # Manejo de dimensiones (seq_len o no)
        if h_context.dim() == 3:
            # Mean pooling sobre secuencia
            h = h_context.mean(dim=1)  # [batch, input_dim]
        else:
            h = h_context
        
        # Input projection
        x = self.input_proj(h)
        x = self.input_norm(x)
        x = F.gelu(x)
        
        # Residual blocks
        for block in self.blocks:
            residual = x
            x = block(x)
            x = x + residual  # Skip connection
            x = F.gelu(x)
        
        # Output projection
        z_pred = self.output_proj(x)
        
        return z_pred

## 🧮 Paso 3: VICReg Loss con SIGReg Weighting

### Función de Pérdida Matemática

$$L_{Total} = \mathbb{E}_{(x,y) \sim D} [ w_{SIG}(y) \cdot L_{VICReg}(z_x, z_y) ]$$

Donde:

$$L_{VICReg} = \lambda L_{Inv} + \mu L_{Var} + \nu L_{Cov}$$

**Componentes:**

1. **Invariance:** $L_{Inv} = \frac{1}{N} \sum_{i=1}^{N} \| z_{x}^{(i)} - z_{y}^{(i)} \|_2^2$

2. **Variance:** $L_{Var} = \frac{1}{d} \sum_{j=1}^{d} \max(0, \gamma - \text{std}(z^{(j)}))$

3. **Covariance:** $L_{Cov} = \frac{1}{d} \sum_{i \neq j} [C(Z)]_{i,j}^2$


In [ ]:
class VICRegSIGRegLoss(nn.Module):
    """VICReg loss con ponderación SIGReg para prevenir colapso dimensional.
    
    Args:
        lambda_inv: Peso de invariance loss (similitud semántica)
        mu_var: Peso de variance loss (prevención de colapso)
        nu_cov: Peso de covariance loss (descorrelación)
        gamma: Target std para variance regularization
        alpha: Peso base para SIGReg (balance entre todos los ejemplos)
    """
    
    def __init__(
        self,
        lambda_inv: float = 25.0,
        mu_var: float = 25.0,
        nu_cov: float = 1.0,
        gamma: float = 1.0,
        alpha: float = 0.3,
        eps: float = 1e-4,
    ):
        super().__init__()
        self.lambda_inv = lambda_inv
        self.mu_var = mu_var
        self.nu_cov = nu_cov
        self.gamma = gamma
        self.alpha = alpha
        self.eps = eps
    
    def invariance_loss(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """L_Inv: MSE entre predicción y target."""
        return F.mse_loss(z1, z2)
    
    def variance_loss(self, z: torch.Tensor) -> torch.Tensor:
        """L_Var: Penaliza desviaciones estándar por debajo de gamma."""
        # z: [batch_size, dim]
        std = torch.sqrt(z.var(dim=0) + self.eps)  # [dim]
        loss = torch.mean(F.relu(self.gamma - std))
        return loss
    
    def covariance_loss(self, z: torch.Tensor) -> torch.Tensor:
        """L_Cov: Penaliza correlaciones off-diagonal."""
        # z: [batch_size, dim]
        batch_size, dim = z.shape
        
        # Centrar los datos
        z = z - z.mean(dim=0, keepdim=True)
        
        # Matriz de covarianza [dim, dim]
        cov_matrix = (z.T @ z) / (batch_size - 1)
        
        # Penalizar solo elementos off-diagonal
        # Método eficiente: suma total - traza
        cov_loss = (cov_matrix ** 2).sum() - (torch.diagonal(cov_matrix) ** 2).sum()
        cov_loss = cov_loss / dim
        
        return cov_loss
    
    def sigreg_weight(
        self, 
        logits: torch.Tensor, 
        target_ids: torch.Tensor
    ) -> torch.Tensor:
        """Calcula peso SIGReg basado en self-information.
        
        Args:
            logits: [batch, seq_len, vocab_size] del Teacher
            target_ids: [batch, seq_len] IDs de tokens del target
        
        Returns:
            weights: [batch] pesos por ejemplo
        """
        batch_size, seq_len, vocab_size = logits.shape
        
        # Calcular probabilidades
        log_probs = F.log_softmax(logits, dim=-1)  # [batch, seq, vocab]
        
        # Recolectar probabilidades de los tokens target
        # target_ids: [batch, seq_len]
        target_log_probs = torch.gather(
            log_probs, 
            dim=-1, 
            index=target_ids.unsqueeze(-1)
        ).squeeze(-1)  # [batch, seq_len]
        
        # Self-information promedio por ejemplo
        # -log P(y) es alto para tokens raros/informativos
        self_info = -target_log_probs.mean(dim=1)  # [batch]
        
        # Normalizar con sigmoid para estabilidad
        normalized_info = torch.sigmoid(self_info)
        
        # Combinar con alpha base
        weights = self.alpha + (1 - self.alpha) * normalized_info
        
        return weights
    
    def forward(
        self,
        z_pred: torch.Tensor,
        z_target: torch.Tensor,
        teacher_logits: Optional[torch.Tensor] = None,
        target_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """Calcula la pérdida total con métricas de monitoreo.
        
        Args:
            z_pred: [batch, dim] predicciones del Predictor
            z_target: [batch, dim] embeddings del Teacher (stop-gradient ya aplicado)
            teacher_logits: [batch, seq, vocab] para SIGReg (opcional)
            target_ids: [batch, seq] para SIGReg (opcional)
        
        Returns:
            loss: Pérdida total escalar
            metrics: Diccionario de componentes para logging
        """
        # Componentes de VICReg
        l_inv = self.invariance_loss(z_pred, z_target)
        
        # Variance en ambas ramas (estabilidad)
        l_var_pred = self.variance_loss(z_pred)
        l_var_target = self.variance_loss(z_target)
        l_var = (l_var_pred + l_var_target) / 2
        
        # Covariance en ambas ramas
        l_cov_pred = self.covariance_loss(z_pred)
        l_cov_target = self.covariance_loss(z_target)
        l_cov = (l_cov_pred + l_cov_target) / 2
        
        # VICReg base
        vicreg_loss = (
            self.lambda_inv * l_inv +
            self.mu_var * l_var +
            self.nu_cov * l_cov
        )
        
        # Aplicar SIGReg si se proporciona información de logits
        if teacher_logits is not None and target_ids is not None:
            weights = self.sigreg_weight(teacher_logits, target_ids)  # [batch]
            # Promedio ponderado
            loss = (vicreg_loss * weights).mean()
        else:
            weights = torch.ones(z_pred.size(0), device=z_pred.device)
            loss = vicreg_loss
        
        # Métricas de monitoreo
        with torch.no_grad():
            metrics = {
                'loss/total': loss.item(),
                'loss/invariance': l_inv.item(),
                'loss/variance': l_var.item(),
                'loss/covariance': l_cov.item(),
                'metrics/z_pred_std': z_pred.std().item(),
                'metrics/z_target_std': z_target.std().item(),
                'metrics/z_pred_norm': z_pred.norm(dim=-1).mean().item(),
                'metrics/sigreg_weight_mean': weights.mean().item(),
            }
        
        return loss, metrics

## 🏛️ Paso 4: Arquitectura JEPA Completa (Dual-Tower)

### Diseño Asimétrico

```
┌─────────────────────────────────────────────┐
│            Context (x)     Target (y)       │
│                 ↓               ↓           │
│         ┌───────────┐   ┌───────────┐      │
│         │  Student  │   │  Teacher  │      │
│         │ (θ train) │   │ (φ EMA)   │      │
│         └─────┬─────┘   └─────┬─────┘      │
│               │               │             │
│              h_x          h_y (stop_grad)   │
│               │               │             │
│         ┌─────▼─────┐         │             │
│         │ Predictor │         │             │
│         │  (ψ)      │         │             │
│         └─────┬─────┘         │             │
│               │               │             │
│             z_pred         z_target         │
│               └───────┬───────┘             │
│                       ↓                     │
│                  VICReg Loss                │
└─────────────────────────────────────────────┘
```


In [ ]:
class QwenJEPA(nn.Module):
    """Joint-Embedding Predictive Architecture basada en Qwen 2.5.
    
    Componentes:
    - Student (Context Encoder): Entrenable
    - Teacher (Target Encoder): Actualización EMA (no gradientes)
    - Predictor: Mapea h_x → z_pred (espacio de h_y)
    """
    
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-0.5B",
        predictor_hidden_dim: int = 4096,
        predictor_depth: int = 3,
        ema_momentum: float = 0.996,
    ):
        super().__init__()
        
        # Cargar modelo base
        print(f"📥 Cargando {model_name}...")
        config = AutoConfig.from_pretrained(model_name)
        self.hidden_dim = config.hidden_size  # 1024 para Qwen 0.5B
        
        # Student (Context Encoder) - Entrenable
        self.student = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        )
        
        # Teacher (Target Encoder) - Solo EMA
        self.teacher = deepcopy(self.student)
        self.teacher.requires_grad_(False)  # Stop gradient
        
        # Predictor (The Bridge)
        self.predictor = LatentPredictor(
            input_dim=self.hidden_dim,
            hidden_dim=predictor_hidden_dim,
            num_blocks=predictor_depth,
        )
        
        # EMA momentum (se actualizará con schedule coseno)
        self.register_buffer('ema_momentum', torch.tensor(ema_momentum))
        
        print(f"✅ JEPA inicializado:")
        print(f"   - Hidden dim: {self.hidden_dim}")
        print(f"   - Predictor hidden: {predictor_hidden_dim}")
        print(f"   - Predictor depth: {predictor_depth}")
        print(f"   - EMA momentum: {ema_momentum}")
    
    @torch.no_grad()
    def update_teacher_ema(self):
        """Actualiza Teacher usando Exponential Moving Average.
        
        φ ← m·φ + (1-m)·θ
        
        Debe llamarse después de cada step del optimizador.
        """
        m = self.ema_momentum.item()
        
        for param_student, param_teacher in zip(
            self.student.parameters(), 
            self.teacher.parameters()
        ):
            param_teacher.data.mul_(m).add_(param_student.data, alpha=1 - m)
    
    def update_ema_momentum(self, step: int, max_steps: int, base_momentum: float = 0.996):
        """Actualiza momentum con schedule coseno.
        
        m_t = 1 - (1 - m_base) * 0.5 * (1 + cos(πt/T))
        
        Args:
            step: Paso actual del entrenamiento
            max_steps: Total de pasos de entrenamiento
            base_momentum: Valor base de momentum
        """
        progress = step / max_steps
        momentum = 1 - (1 - base_momentum) * 0.5 * (1 + math.cos(math.pi * progress))
        self.ema_momentum.fill_(momentum)
    
    def encode_context(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Codifica contexto con Student (entrenable).
        
        Returns:
            h_x: [batch, hidden_dim] representación del contexto
        """
        outputs = self.student(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        # Usar último hidden state + mean pooling
        hidden_states = outputs.last_hidden_state  # [batch, seq, hidden]
        
        # Mean pooling (atendiendo a mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        h_x = sum_embeddings / sum_mask  # [batch, hidden_dim]
        
        return h_x
    
    @torch.no_grad()
    def encode_target(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Codifica target con Teacher (no gradientes).
        
        Returns:
            h_y: [batch, hidden_dim] representación del target
            logits: [batch, seq, vocab] para SIGReg
        """
        outputs = self.teacher(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        hidden_states = outputs.last_hidden_state
        
        # Mean pooling
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        h_y = sum_embeddings / sum_mask
        
        # Logits para SIGReg (proyección a vocabulario)
        # Usamos el embedding layer transpuesto
        logits = torch.matmul(
            hidden_states, 
            self.teacher.embed_tokens.weight.T
        )  # [batch, seq, vocab]
        
        return h_y, logits
    
    def forward(
        self,
        context_ids: torch.Tensor,
        context_mask: torch.Tensor,
        target_ids: torch.Tensor,
        target_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass completo de JEPA.
        
        Args:
            context_ids: [batch, seq_ctx] tokens del contexto
            context_mask: [batch, seq_ctx] máscara de atención
            target_ids: [batch, seq_tgt] tokens del target
            target_mask: [batch, seq_tgt] máscara de atención
        
        Returns:
            z_pred: [batch, hidden_dim] predicción
            z_target: [batch, hidden_dim] target (stop-gradient)
            teacher_logits: [batch, seq_tgt, vocab] para SIGReg
        """
        # Codificar contexto con Student
        h_context = self.encode_context(context_ids, context_mask)
        
        # Predecir representación del target
        z_pred = self.predictor(h_context)
        
        # Codificar target con Teacher (no gradientes)
        z_target, teacher_logits = self.encode_target(target_ids, target_mask)
        
        return z_pred, z_target, teacher_logits

## 📊 Paso 5: Data Pipeline con Block Masking

### Estrategia de Masking Jerárquico

```
Documento completo:
[████████████████████████████████████████]

Context (70%):
[████████████████████████████░░░░░░░░░░░░]
                            ↑
                    Student ve esto

Target (20%):
[░░░░░░░░░░░░░░░░░░░░░░░░░░░░████████░░░░]
                            ↑
                    Predictor debe inferir esto
```


In [ ]:
@dataclass
class JEPADataConfig:
    """Configuración del pipeline de datos."""
    max_length: int = 512  # Longitud máxima de secuencia
    context_ratio: float = 0.7  # 70% inicial para contexto
    target_ratio: float = 0.2  # 20% para target
    min_target_length: int = 50  # Mínimo de tokens en target
    batch_size: int = 8  # Batch size base
    gradient_accumulation_steps: int = 4  # Batch efectivo = 32


class JEPAStreamingDataset(IterableDataset):
    """Dataset streaming con block masking para JEPA.
    
    Ventajas:
    - No descarga datos al disco (streaming=True)
    - Block masking contiguos (no random token masking)
    - Balanceo dinámico de context/target
    """
    
    def __init__(
        self,
        tokenizer,
        config: JEPADataConfig,
        dataset_name: str = "HuggingFaceFW/fineweb-edu",
        dataset_config: str = "sample-10BT",
    ):
        self.tokenizer = tokenizer
        self.config = config
        
        # Cargar dataset en modo streaming
        print(f"📡 Iniciando streaming de {dataset_name}/{dataset_config}...")
        self.dataset = load_dataset(
            dataset_name,
            dataset_config,
            split="train",
            streaming=True,
        )
        
        # Shuffle para diversidad
        self.dataset = self.dataset.shuffle(seed=42, buffer_size=10000)
    
    def create_context_target_split(self, tokens: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Divide secuencia en contexto y target con block masking.
        
        Args:
            tokens: [seq_len] secuencia tokenizada
        
        Returns:
            context_tokens: [context_len] primeros 70%
            target_tokens: [target_len] bloque del 20% restante
        """
        seq_len = len(tokens)
        
        # Calcular puntos de corte
        context_end = int(seq_len * self.config.context_ratio)
        target_start = context_end
        target_end = min(
            target_start + int(seq_len * self.config.target_ratio),
            seq_len
        )
        
        # Asegurar longitud mínima del target
        if target_end - target_start < self.config.min_target_length:
            target_end = min(target_start + self.config.min_target_length, seq_len)
        
        context_tokens = tokens[:context_end]
        target_tokens = tokens[target_start:target_end]
        
        return context_tokens, target_tokens
    
    def __iter__(self):
        """Genera ejemplos (context, target) infinitamente."""
        for example in self.dataset:
            text = example['text']
            
            # Tokenizar
            encoded = self.tokenizer(
                text,
                max_length=self.config.max_length,
                truncation=True,
                return_tensors='pt',
            )
            
            input_ids = encoded['input_ids'].squeeze(0)
            
            # Filtrar secuencias muy cortas
            if len(input_ids) < self.config.min_target_length * 2:
                continue
            
            # Crear split context/target
            context_ids, target_ids = self.create_context_target_split(input_ids)
            
            yield {
                'context_ids': context_ids,
                'target_ids': target_ids,
            }


def jepa_collate_fn(batch, tokenizer):
    """Collate function con padding dinámico."""
    context_ids = [item['context_ids'] for item in batch]
    target_ids = [item['target_ids'] for item in batch]
    
    # Padding con tokenizer
    context_batch = tokenizer.pad(
        {'input_ids': context_ids},
        padding=True,
        return_tensors='pt',
    )
    
    target_batch = tokenizer.pad(
        {'input_ids': target_ids},
        padding=True,
        return_tensors='pt',
    )
    
    return {
        'context_ids': context_batch['input_ids'],
        'context_mask': context_batch['attention_mask'],
        'target_ids': target_batch['input_ids'],
        'target_mask': target_batch['attention_mask'],
    }

## 🎓 Paso 6: Loop de Entrenamiento con Mixed Precision

### Optimizaciones para Colab Free Tier

1. **Mixed Precision (FP16):** 2x velocidad, 50% menos VRAM
2. **Gradient Accumulation:** Batch efectivo grande sin OOM
3. **Gradient Clipping:** Estabilidad en VICReg
4. **EMA Scheduling:** Convergencia suave del Teacher


In [ ]:
def train_jepa(
    model: QwenJEPA,
    criterion: VICRegSIGRegLoss,
    train_dataloader: DataLoader,
    num_steps: int = 10000,
    learning_rate: float = 1e-4,
    gradient_accumulation_steps: int = 4,
    gradient_clip_val: float = 1.0,
    log_interval: int = 100,
    save_interval: int = 2000,
    save_path: str = "./checkpoints",
):
    """Loop de entrenamiento principal para JEPA.
    
    Args:
        model: Modelo QwenJEPA
        criterion: Función de pérdida VICReg+SIGReg
        train_dataloader: DataLoader con ejemplos (context, target)
        num_steps: Total de pasos de optimización
        learning_rate: Learning rate base
        gradient_accumulation_steps: Acumulación de gradientes
        gradient_clip_val: Valor máximo de norma de gradiente
        log_interval: Intervalo de logging a WandB
        save_interval: Intervalo de guardado de checkpoints
        save_path: Ruta para guardar checkpoints
    """
    import os
    from pathlib import Path
    
    # Crear directorio de checkpoints
    Path(save_path).mkdir(parents=True, exist_ok=True)
    
    # Optimizer: Solo Student + Predictor son entrenables
    trainable_params = [
        {'params': model.student.parameters(), 'lr': learning_rate},
        {'params': model.predictor.parameters(), 'lr': learning_rate * 10},  # Predictor aprende más rápido
    ]
    
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=learning_rate,
        betas=(0.9, 0.999),
        weight_decay=0.01,
    )
    
    # Learning rate scheduler (cosine annealing)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_steps,
        eta_min=learning_rate * 0.01,
    )
    
    # Scaler para mixed precision
    scaler = GradScaler()
    
    # Mover modelo a GPU
    model = model.to(device)
    model.train()
    
    # Logging inicial
    print(f"🚀 Iniciando entrenamiento JEPA")
    print(f"   - Total steps: {num_steps}")
    print(f"   - Batch size efectivo: {train_dataloader.batch_size * gradient_accumulation_steps}")
    print(f"   - Learning rate: {learning_rate}")
    print(f"   - Device: {device}")
    
    # Training loop
    global_step = 0
    optimizer.zero_grad()
    
    dataloader_iter = iter(train_dataloader)
    
    while global_step < num_steps:
        for accum_step in range(gradient_accumulation_steps):
            try:
                batch = next(dataloader_iter)
            except StopIteration:
                # Reiniciar iterador si se acaba el dataset
                dataloader_iter = iter(train_dataloader)
                batch = next(dataloader_iter)
            
            # Mover batch a GPU
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass con mixed precision
            with autocast():
                z_pred, z_target, teacher_logits = model(
                    context_ids=batch['context_ids'],
                    context_mask=batch['context_mask'],
                    target_ids=batch['target_ids'],
                    target_mask=batch['target_mask'],
                )
                
                # Calcular pérdida
                loss, metrics = criterion(
                    z_pred=z_pred,
                    z_target=z_target,
                    teacher_logits=teacher_logits,
                    target_ids=batch['target_ids'],
                )
                
                # Normalizar por gradient accumulation
                loss = loss / gradient_accumulation_steps
            
            # Backward pass
            scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=gradient_clip_val,
        )
        
        # Optimizer step
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        # Actualizar Teacher con EMA
        model.update_teacher_ema()
        
        # Actualizar EMA momentum
        model.update_ema_momentum(global_step, num_steps)
        
        # Learning rate schedule
        scheduler.step()
        
        global_step += 1
        
        # Logging
        if global_step % log_interval == 0:
            metrics['train/learning_rate'] = scheduler.get_last_lr()[0]
            metrics['train/ema_momentum'] = model.ema_momentum.item()
            metrics['train/grad_norm'] = grad_norm.item()
            
            wandb.log(metrics, step=global_step)
            
            print(f"Step {global_step}/{num_steps} | Loss: {metrics['loss/total']:.4f} | "
                  f"Inv: {metrics['loss/invariance']:.4f} | "
                  f"Var: {metrics['loss/variance']:.4f} | "
                  f"Cov: {metrics['loss/covariance']:.4f}")
        
        # Checkpoint saving
        if global_step % save_interval == 0:
            checkpoint_path = os.path.join(save_path, f"checkpoint_step_{global_step}.pt")
            torch.save({
                'step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'metrics': metrics,
            }, checkpoint_path)
            print(f"💾 Checkpoint guardado en {checkpoint_path}")
    
    print(f"✅ Entrenamiento completado ({num_steps} steps)")

## 🚀 Paso 7: Ejecución del MVP

### Configuración de Experimento


In [ ]:
# Configuración de hiperparámetros
data_config = JEPADataConfig(
    max_length=512,
    context_ratio=0.7,
    target_ratio=0.2,
    min_target_length=50,
    batch_size=8,  # Base batch size
    gradient_accumulation_steps=4,  # Batch efectivo = 32
)

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer.pad_token = tokenizer.eos_token

# Crear dataset
train_dataset = JEPAStreamingDataset(
    tokenizer=tokenizer,
    config=data_config,
)

# Crear dataloader
from functools import partial

train_dataloader = DataLoader(
    train_dataset,
    batch_size=data_config.batch_size,
    collate_fn=partial(jepa_collate_fn, tokenizer=tokenizer),
    num_workers=2,
    pin_memory=True,
)

print("✅ Data pipeline configurado")

In [ ]:
# Inicializar modelo JEPA
model = QwenJEPA(
    model_name="Qwen/Qwen2.5-0.5B",
    predictor_hidden_dim=4096,
    predictor_depth=3,
    ema_momentum=0.996,
)

# Inicializar función de pérdida
criterion = VICRegSIGRegLoss(
    lambda_inv=25.0,
    mu_var=25.0,
    nu_cov=1.0,
    gamma=1.0,
    alpha=0.3,
)

print("✅ Modelo y criterio inicializados")

# Imprimir estadísticas del modelo
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Parámetros totales: {total_params:,}")
print(f"📊 Parámetros entrenables: {trainable_params:,}")

In [ ]:
# Lanzar entrenamiento
train_jepa(
    model=model,
    criterion=criterion,
    train_dataloader=train_dataloader,
    num_steps=10000,  # MVP: 10k steps (~6-8 horas en T4)
    learning_rate=1e-4,
    gradient_accumulation_steps=data_config.gradient_accumulation_steps,
    gradient_clip_val=1.0,
    log_interval=100,
    save_interval=2000,
    save_path="./checkpoints",
)

## 📊 Paso 8: Evaluación y Visualización

### Métricas de Calidad del Espacio Latente


In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

@torch.no_grad()
def evaluate_latent_space(
    model: QwenJEPA,
    eval_dataloader: DataLoader,
    num_samples: int = 1000,
):
    """Evalúa la calidad del espacio latente aprendido.
    
    Métricas:
    1. Variance de embeddings (debe estar cerca de γ=1)
    2. Covariance entropy (descorrelación)
    3. Collapse metric (norma promedio)
    4. T-SNE visualization
    """
    model.eval()
    model = model.to(device)
    
    all_z_pred = []
    all_z_target = []
    
    print(f"🔍 Evaluando espacio latente con {num_samples} muestras...")
    
    for i, batch in enumerate(eval_dataloader):
        if i * eval_dataloader.batch_size >= num_samples:
            break
        
        batch = {k: v.to(device) for k, v in batch.items()}
        
        z_pred, z_target, _ = model(
            context_ids=batch['context_ids'],
            context_mask=batch['context_mask'],
            target_ids=batch['target_ids'],
            target_mask=batch['target_mask'],
        )
        
        all_z_pred.append(z_pred.cpu())
        all_z_target.append(z_target.cpu())
    
    z_pred_cat = torch.cat(all_z_pred, dim=0)
    z_target_cat = torch.cat(all_z_target, dim=0)
    
    # Métrica 1: Variance
    var_pred = z_pred_cat.var(dim=0).mean().item()
    var_target = z_target_cat.var(dim=0).mean().item()
    
    # Métrica 2: Norma promedio (collapse detector)
    norm_pred = z_pred_cat.norm(dim=-1).mean().item()
    norm_target = z_target_cat.norm(dim=-1).mean().item()
    
    # Métrica 3: Covariance matrix
    z_centered = z_pred_cat - z_pred_cat.mean(dim=0)
    cov_matrix = (z_centered.T @ z_centered) / (z_pred_cat.size(0) - 1)
    
    # Covariance entropy
    cov_diag = torch.diagonal(cov_matrix)
    cov_off_diag = cov_matrix - torch.diag(cov_diag)
    off_diag_power = (cov_off_diag ** 2).sum().item()
    
    print(f"\n📈 Métricas del Espacio Latente:")
    print(f"   Variance (pred): {var_pred:.4f} (target: 1.0)")
    print(f"   Variance (target): {var_target:.4f}")
    print(f"   Norma L2 (pred): {norm_pred:.4f}")
    print(f"   Norma L2 (target): {norm_target:.4f}")
    print(f"   Off-diagonal power: {off_diag_power:.4f} (menor es mejor)")
    
    # Visualización T-SNE
    print(f"\n🎨 Generando visualización T-SNE...")
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    z_combined = torch.cat([z_pred_cat[:500], z_target_cat[:500]], dim=0).numpy()
    z_2d = tsne.fit_transform(z_combined)
    
    plt.figure(figsize=(12, 6))
    
    # Plot predicciones
    plt.subplot(1, 2, 1)
    plt.scatter(z_2d[:500, 0], z_2d[:500, 1], alpha=0.6, s=5, c='blue', label='Predicciones')
    plt.title('Espacio Latente - Predicciones')
    plt.xlabel('T-SNE Dimension 1')
    plt.ylabel('T-SNE Dimension 2')
    plt.legend()
    
    # Plot targets
    plt.subplot(1, 2, 2)
    plt.scatter(z_2d[500:, 0], z_2d[500:, 1], alpha=0.6, s=5, c='red', label='Targets')
    plt.title('Espacio Latente - Targets')
    plt.xlabel('T-SNE Dimension 1')
    plt.ylabel('T-SNE Dimension 2')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('latent_space_tsne.png', dpi=300, bbox_inches='tight')
    wandb.log({"latent_space_visualization": wandb.Image('latent_space_tsne.png')})
    plt.show()
    
    print(f"✅ Evaluación completada")
    
    return {
        'variance_pred': var_pred,
        'variance_target': var_target,
        'norm_pred': norm_pred,
        'norm_target': norm_target,
        'off_diagonal_power': off_diag_power,
    }

In [ ]:
# Ejecutar evaluación
eval_metrics = evaluate_latent_space(
    model=model,
    eval_dataloader=train_dataloader,
    num_samples=1000,
)

# Log métricas finales
wandb.log({f"eval/{k}": v for k, v in eval_metrics.items()})

## 💾 Paso 9: Guardar Modelo en Hugging Face Hub


In [ ]:
from huggingface_hub import HfApi, create_repo

# Configuración
repo_name = "your-username/qwen-jepa-0.5b-v1"  # Cambiar por tu nombre de usuario

# Crear repositorio
api = HfApi()
try:
    create_repo(repo_name, repo_type="model", exist_ok=True)
    print(f"✅ Repositorio creado: https://huggingface.co/{repo_name}")
except Exception as e:
    print(f"ℹ️  Repositorio ya existe o error: {e}")

# Guardar modelo completo
model.save_pretrained(f"./{repo_name}")
tokenizer.save_pretrained(f"./{repo_name}")

# Subir a HF Hub
api.upload_folder(
    folder_path=f"./{repo_name}",
    repo_id=repo_name,
    repo_type="model",
)

print(f"🚀 Modelo publicado en Hugging Face Hub")

## 📝 Notas Finales

### ✅ Checklist de Implementación

- [x] Dual-Tower asimétrica (Student/Teacher)
- [x] Latent Predictor con BatchNorm
- [x] VICReg loss (Invariance + Variance + Covariance)
- [x] SIGReg weighting basado en self-information
- [x] EMA coseno para Teacher
- [x] Block masking jerárquico
- [x] Streaming dataset (0 bytes en disco)
- [x] Mixed precision training
- [x] Gradient accumulation
- [x] WandB monitoring
- [x] T-SNE visualization
- [x] Checkpoint saving
- [x] Hugging Face Hub integration

### 🎯 Métricas de Éxito

1. **Variance ~ 1.0:** Espacio latente no colapsado
2. **Off-diagonal power < 10:** Características descorrelacionadas
3. **Invariance loss < 0.5:** Buena alineación semántica
4. **T-SNE clusters:** Separación semántica visual

### 🚀 Próximos Pasos

1. **Fine-tuning:** Adaptar a tareas downstream (clasificación, QA)
2. **Scaling:** Probar con Qwen 1.5B o 3B
3. **Multi-modal:** Extender a visión (I-JEPA)
4. **Deployment:** Quantization y optimización para inferencia

---

**Nomad - Ultra - Think**  
*Arquitectura Cognitiva de Próxima Generación*
